# R0 — TPSMM baseline on LSA64 (M1 gate), session 2 of 2

Target (paper Table 1, LSA64): L1 0.01342 | SSIM 0.9208 | LPIPS 0.02261 |
FVD 182.785 | TCD 0.130.

A full run is **15.5 GPU-hours** (D12, measured at 1.07 s/step) against a ~9h
session cap, so it spans two sessions of ~50 epochs each.

`num_epochs` is a **total**, not an increment: the vendored `train.py` runs
`for epoch in trange(start_epoch, num_epochs)` after `start_epoch = load_cpk() + 1`,
and restores the LR schedule with `last_epoch=start_epoch-1`. Bounding a session
therefore means giving it a lower total.

Bounded rather than left to be killed at the cap: a killed session's output is
not reliably saved, which would cost both the epochs since the last checkpoint
and the run's log.

Logic lives in `pgmm/train/session.py` where it is unit-tested. This is a driver.


In [ ]:
import subprocess, sys, time
import torch

print('torch', torch.__version__, '| gpus', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  [{i}] {p.name}  {p.total_memory/2**30:.1f} GiB')
# Quota bills session wall-clock, not GPU-hours, so a one-GPU session wastes half
# of what it costs. Needs machine_shape=NvidiaTeslaT4 in the metadata (D22).
assert torch.cuda.device_count() == 2, 'expected 2x T4 -- check machine_shape'

r = subprocess.run(['git', 'clone', '-q', '--branch', 'feat/m0-m1-harness',
                    '--depth', '1', 'https://github.com/SonLamHG/pgmm.git',
                    '/kaggle/working/repo'], capture_output=True, text=True)
assert r.returncode == 0, r.stderr
sys.path.insert(0, '/kaggle/working/repo')
print('repo cloned')

In [ ]:
# Extract every mounted kernel output: the prepared data, and -- in session 2 --
# the previous session's checkpoint. Kaggle bundles each kernel's output into a
# single _output_.zip, and that is what kernel_sources mounts (D20).
import zipfile
from pathlib import Path

WORK = Path('/kaggle/working/mounted')
zips = sorted(Path('/kaggle/input').rglob('_output_.zip'))
assert zips, 'nothing mounted -- check kernel_sources'
for z in zips:
    t0 = time.time()
    with zipfile.ZipFile(z) as zf:
        zf.extractall(WORK / z.parent.name)
    print(f'{z.parent.name:24} {z.stat().st_size/2**30:5.2f} GiB '
          f'-> {(time.time()-t0)/60:.1f} min')

In [ ]:
from pgmm.train.session import (build_session_config, find_prepared_data,
                                find_resume_checkpoint, steps_per_epoch)

DATA = find_prepared_data(WORK)
n_train = len(list((DATA / 'train').iterdir()))
n_test = len(list((DATA / 'test').iterdir()))
print('data  :', DATA)
print('split :', n_train, 'train /', n_test, 'test')
# Without train/ and test/ on disk, FramesDataset silently substitutes its own
# random 80/20 split -- 2560/640, not the paper's 2800/400 (D17).
assert (n_train, n_test) == (2800, 400), f'paper says 2800/400, got {n_train}/{n_test}'

RESUME = find_resume_checkpoint(WORK)
print('resume:', RESUME or 'none - starting from scratch')
assert (RESUME is not None) == True, (
    'session 2 expected resume=True, found ' + str(RESUME))

MAX_EPOCHS = 100
CFG = Path('/kaggle/working/session.yaml')
cfg = build_session_config('/kaggle/working/repo/pgmm/config/lsa64-tpsmm.yaml',
                           DATA, CFG, max_epochs=MAX_EPOCHS)
spe = steps_per_epoch(cfg)
print(f'this session -> up to epoch {MAX_EPOCHS} total, {spe:.0f} steps/epoch')
print(f'projected     {MAX_EPOCHS*spe*1.07/3600:.1f}h at the measured 1.07 s/step')

In [ ]:
argv = [sys.executable, 'run.py', '--config', str(CFG),
        '--log_dir', '/kaggle/working/log', '--device_ids', '0,1']
if RESUME is not None:
    argv += ['--checkpoint', str(RESUME)]
print(' '.join(argv))

t0 = time.time()
r = subprocess.run(argv, cwd='/kaggle/working/repo/third_party/tpsmm',
                   capture_output=True, text=True)
elapsed = time.time() - t0
print(r.stdout[-3000:])
print('--- stderr tail ---')
print((r.stderr or '')[-2500:])
print('exit:', r.returncode, f'| wall {elapsed/3600:.2f}h')

In [ ]:
# A number derived from a crash looks like evidence. Refuse to report one.
assert r.returncode == 0, f'training exited {r.returncode} -- see stderr above'

ckpts = sorted(Path('/kaggle/working/log').rglob('*-checkpoint.pth.tar'))
assert ckpts, 'no checkpoint written -- the next session would have nothing to resume from'
print('checkpoints:')
for p in ckpts:
    print(f'  {p.name}  {p.stat().st_size/2**20:.0f} MiB')
print(f'\ns/step this session: {elapsed/(MAX_EPOCHS*spe):.3f}  (D12 measured 1.07)')
for lg in Path('/kaggle/working/log').rglob('log.txt'):
    lines = lg.read_text().strip().splitlines()
    print('first:', lines[0][:110])
    print('last :', lines[-1][:110])

In [ ]:
# The output must carry the checkpoint to the next session and nothing else:
# the extracted data would republish 1.25 GiB and the repo clone would ride along.
import shutil

shutil.rmtree(WORK, ignore_errors=True)
shutil.rmtree('/kaggle/working/repo', ignore_errors=True)
Path('/kaggle/working/session.yaml').unlink(missing_ok=True)
print('output root:', sorted(p.name for p in Path('/kaggle/working').iterdir()))